# Dialogue Generation with LSTM + Attention

This small encoder-decoder learns fixed conversational responses. The attention layer lets the decoder use the encoded dialogue context at every output step.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

pairs = [
	("hello", "hi"), ("how are you", "i am fine"), ("what is your name", "i am a bot"),
	("good morning", "good morning"), ("thank you", "you are welcome"), ("goodbye", "see you")
]
questions = ["<start> " + q + " <end>" for q, _ in pairs]
answers = ["<start> " + a + " <end>" for _, a in pairs]
vocabulary = sorted(set(" ".join(questions + answers).split()))
token_to_id = {token: index + 1 for index, token in enumerate(vocabulary)}
vocab_size = len(token_to_id) + 1

def encode(sentences, length):
	values = [[token_to_id[token] for token in sentence.split()] for sentence in sentences]
	return tf.keras.utils.pad_sequences(values, maxlen=length, padding="post")

max_length = max(len(sentence.split()) for sentence in questions + answers)
encoder_input = encode(questions, max_length)
decoder_input = encode(answers, max_length)
decoder_target = np.roll(decoder_input, -1, axis=1)

encoder_tokens = layers.Input((max_length,))
decoder_tokens = layers.Input((max_length,))
embedding = layers.Embedding(vocab_size, 32, mask_zero=True)
encoder_output, state_h, state_c = layers.LSTM(64, return_sequences=True, return_state=True)(embedding(encoder_tokens))
decoder_output = layers.LSTM(64, return_sequences=True)(embedding(decoder_tokens), initial_state=[state_h, state_c])
decoder_output = layers.Attention()([decoder_output, encoder_output])
decoder_output = layers.Dense(vocab_size, activation="softmax")(decoder_output)
model = models.Model([encoder_tokens, decoder_tokens], decoder_output)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit([encoder_input, decoder_input], decoder_target[..., None], epochs=150, verbose=0)
print("Training complete. Attention output shape:", model.output_shape)

predicted = model.predict([encoder_input[:1], decoder_input[:1]], verbose=0)[0]
predicted_ids = predicted.argmax(axis=-1)
id_to_token = {value: key for key, value in token_to_id.items()}
print("Example response:", " ".join(id_to_token.get(index, "<pad>") for index in predicted_ids if index))